# WEC Performance Analysis - Phase 2 | Stochastic Frontier Analysis (SFA)

Stochastic Frontier Analysis for technical efficiency scoring and mechanical degradation detection across 12 Wave Energy Converters (WECs).

## Motivation
Phase 1 (XGBoost regression) identifies the absolute deviation of each WEC from its model-predicted output. Phase 2 (DEA) identifies inefficiency deterministically: every deviation from the empirical frontier is labelled as waste. In a marine environment this is a serious flaw because random sea-state variation, sensor noise, and measurement error are genuinely symmetric and cannot be attributed to the asset.

SFA addresses this by decomposing the composite residual into two statistically distinct components:
`epsilon_i = v_i - u_i`

where:
* `v_i ~ N(0, sigma_v^2)` symmetric noise (waves, sensors)
* `u_i ~ |N(0, sigma_u^2)|` one-sided technical inefficiency

## Model specification: 1D Cobb-Douglas (log-linear)
`ln(Y_i) = beta_0 + beta_1 * ln(WPF_i) + v_i - u_i`

Parameters estimated by Maximum Likelihood Estimation (MLE):
* `beta_0`: intercept
* `beta_1`: output elasticity with respect to wave power flux
* `lambda`: sigma_u / sigma_v (signal-to-noise ratio)
* `sigma^2`: sigma_u^2 + sigma_v^2 (total error variance)

## Pipeline
1. `load_and_prepare` -- ingest CSV, compute WPF, apply log transform
2. `fit_sfa_epoch1` -- MLE on Epoch 1 only
3. `score_efficiency` -- Battese-Coelli estimator on full dataset
4. `compute_generation_deficit` -- back-transform frontier, compute kW deficit
5. `aggregate_rolling` -- 7-day rolling mean per buoy
6. `plot_timeseries` -- efficiency curves for 12 buoys
7. `plot_residual_decomp` -- KDE separation of v and u for Epoch 3
8. `plot_triple_frontier` -- 3-panel frontier scatter (1x3, Epoch 3)
9. `print_degradation_report` -- enhanced terminal report with deficit ranking

In [1]:
from __future__ import annotations

import logging
import os
import warnings
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import matplotlib
# matplotlib.use("Agg") # Comentado para permitir que os graficos aparecam no Jupyter Notebook
%matplotlib inline 
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.optimize import minimize, OptimizeResult
from scipy.stats import norm

# ---------------------------------------------------------------------------
# Logging
# ---------------------------------------------------------------------------
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(message)s",
    datefmt="%Y-%m-%d %H:%M:%S",
)
logger = logging.getLogger("Phase2_SFA")
warnings.filterwarnings("ignore")

# ---------------------------------------------------------------------------
# Constants
# ---------------------------------------------------------------------------
DATA_PATH: str = "dataset2/wec_c5_mock_data_epochs.csv"
TIMESTAMP_COL: str = "PCTimeStamp"
BUOY_COL: str = "Buoy_ID"
TARGET_COL: str = "Energy_Generation_kW"
WPF_COL: str = "Wave_Power_Flux"
EPOCH_COL: str = "Epoch_Marker"

OUTPUT_CAP: float = 350.0       # physical rated capacity of each WEC [kW]
LOG_EPS: float = 1e-6           # guard constant added before log
ROLLING_WINDOW: str = "7D"      # smoothing window for efficiency time series

HEALTHY_BUOYS: List[str] = [f"Boia_{i}" for i in range(1, 9)]
DEGRADED_BUOYS: List[str] = [f"Boia_{i}" for i in range(9, 13)]
ALL_BUOYS: List[str] = HEALTHY_BUOYS + DEGRADED_BUOYS

PLOT_DIR: Path = Path("plots/phase2_SFA/")
PHASE2_CSV_OUT: str = "dataset2/wec_phase2_outputs.csv"

# Colour palette used consistently across all plots
COLOR_HEALTHY: str = "#2471A3"
COLOR_DEGRADED: str = "#C0392B"
COLOR_FRONTIER: str = "#1A5276"

## Core Functions
Definição das funções para modelação SFA, Maximum Likelihood Estimation (MLE), scoring de eficiência de Battese-Coelli e visualizações.

In [2]:
# ===========================================================================
# Section 1 -- Data Loading and Preparation
# ===========================================================================
def load_and_prepare(csv_path: str) -> pd.DataFrame:
    logger.info("Loading data from: %s", csv_path)
    df: pd.DataFrame = pd.read_csv(csv_path, parse_dates=[TIMESTAMP_COL])
    logger.info("Raw shape: %s", df.shape)

    if WPF_COL not in df.columns:
        logger.info("Column '%s' not found -- computing from Hs and Te", WPF_COL)
        df[WPF_COL] = 0.49 * df["Hs__m"] ** 2 * df["Te__s"]

    df[TARGET_COL] = df[TARGET_COL].clip(upper=OUTPUT_CAP)

    n_before: int = len(df)
    df = df[(df[WPF_COL] > 0) & (df[TARGET_COL] > 0)].copy()
    n_dropped: int = n_before - len(df)
    if n_dropped > 0:
        logger.warning("Dropped %d rows with non-positive WPF or output", n_dropped)

    for col in [WPF_COL, TARGET_COL]:
        n_nan: int = int(df[col].isna().sum())
        if n_nan > 0:
            df[col] = df[col].fillna(df[col].median())
            logger.info("Imputed %d NaN values in column '%s'", n_nan, col)

    df["ln_wpf"] = np.log(df[WPF_COL] + LOG_EPS)
    df["ln_y"] = np.log(df[TARGET_COL] + LOG_EPS)

    df = df.sort_values([TIMESTAMP_COL, BUOY_COL]).reset_index(drop=True)
    logger.info("Prepared shape: %s | Epochs present: %s", df.shape, sorted(df[EPOCH_COL].unique()))
    return df

# ===========================================================================
# Section 2 -- MLE Fitting on Epoch 1
# ===========================================================================
def _neg_log_likelihood(params: np.ndarray, ln_x: np.ndarray, ln_y: np.ndarray) -> float:
    beta0: float = params[0]
    beta1: float = params[1]
    sigma2: float = np.exp(params[2])
    lam: float = np.exp(params[3])

    sigma: float = np.sqrt(sigma2)
    epsilon: np.ndarray = ln_y - beta0 - beta1 * ln_x
    n: int = len(epsilon)

    z: np.ndarray = -lam * epsilon / sigma
    log_phi: np.ndarray = norm.logcdf(z)

    log_lik: float = (
        n * np.log(2)
        - n * np.log(sigma)
        - n * 0.5 * np.log(2.0 * np.pi)
        + log_phi.sum()
        - (epsilon ** 2).sum() / (2.0 * sigma2)
    )
    return -log_lik

def fit_sfa_epoch1(df: pd.DataFrame) -> Dict:
    df_e1: pd.DataFrame = df[(df[EPOCH_COL] == 1) & (df[TARGET_COL] < 345.0)].copy()
    ln_x: np.ndarray = df_e1["ln_wpf"].values
    ln_y: np.ndarray = df_e1["ln_y"].values

    logger.info("Fitting SFA on Epoch 1 Ramp-up Region: %d observations from %d buoys", len(ln_x), df_e1[BUOY_COL].nunique())

    initial_points: List[List[float]] = [
        [3.0, 0.8, np.log(0.5), np.log(1.0)],
        [2.5, 0.9, np.log(0.2), np.log(2.0)],
        [3.5, 0.7, np.log(1.0), np.log(0.5)],
        [3.0, 1.0, np.log(0.1), np.log(3.0)],
    ]

    best_result: Optional[OptimizeResult] = None
    best_nll: float = np.inf

    for x0 in initial_points:
        try:
            res: OptimizeResult = minimize(
                _neg_log_likelihood,
                x0=np.array(x0),
                args=(ln_x, ln_y),
                method="L-BFGS-B",
                options={"maxiter": 5000, "ftol": 1e-12, "gtol": 1e-8},
            )
            if res.fun < best_nll:
                best_nll = res.fun
                best_result = res
        except Exception as exc:
            logger.warning("Optimisation failed for starting point %s: %s", x0, exc)

    if best_result is None or not best_result.success:
        logger.warning("MLE did not converge cleanly -- check model or data quality")

    beta0: float = best_result.x[0]
    beta1: float = best_result.x[1]
    sigma2: float = np.exp(best_result.x[2])
    lam: float = np.exp(best_result.x[3])

    sigma_u2: float = sigma2 * lam ** 2 / (1.0 + lam ** 2)
    sigma_v2: float = sigma2 * 1.0 / (1.0 + lam ** 2)
    sigma_star2: float = sigma_u2 * sigma_v2 / sigma2

    params: Dict = {
        "beta0": beta0, "beta1": beta1, "sigma2": sigma2,
        "lambda_": lam, "sigma_u2": sigma_u2, "sigma_v2": sigma_v2,
        "sigma_star2": sigma_star2, "converged": best_result.success, "nll": best_nll,
    }

    logger.info("MLE results (Epoch 1 frontier):")
    logger.info("  beta_0    = %+.6f", beta0)
    logger.info("  beta_1    = %+.6f  (output elasticity)", beta1)
    logger.info("  lambda    = %.6f   (sigma_u / sigma_v)", lam)
    logger.info("  sigma^2   = %.6f   (total error variance)", sigma2)
    logger.info("  sigma_u^2 = %.6f   (inefficiency variance)", sigma_u2)
    logger.info("  sigma_v^2 = %.6f   (noise variance)", sigma_v2)
    logger.info("  converged = %s | NLL = %.4f", best_result.success, best_nll)

    return params

# ===========================================================================
# Section 3 -- Efficiency Scoring
# ===========================================================================
def score_efficiency(df: pd.DataFrame, params: Dict) -> pd.DataFrame:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]
    sigma2: float = params["sigma2"]
    sigma_u2: float = params["sigma_u2"]
    sigma_star2: float = params["sigma_star2"]
    sigma_star: float = np.sqrt(sigma_star2)

    epsilon: np.ndarray = df["ln_y"].values - beta0 - beta1 * df["ln_wpf"].values
    mu_star: np.ndarray = -epsilon * sigma_u2 / sigma2

    ratio: np.ndarray = mu_star / sigma_star
    te: np.ndarray = (
        np.exp(-mu_star + sigma_star2 / 2.0)
        * norm.cdf(ratio - sigma_star)
        / np.maximum(norm.cdf(ratio), 1e-15)
    )
    te = np.clip(te, 0.0, 1.0)

    df = df.copy()
    df["epsilon"] = epsilon
    df["mu_star"] = mu_star
    df["sigma_noise_hat"] = epsilon - (-mu_star)
    df["SFA_Efficiency"] = te

    logger.info(
        "Efficiency scoring complete | mean TE = %.4f | min = %.4f | max = %.4f",
        te.mean(), te.min(), te.max()
    )
    
    summary = (
        df.groupby([EPOCH_COL, BUOY_COL])["SFA_Efficiency"]
        .mean()
        .unstack(BUOY_COL)
        .round(4)
    )
    logger.info("Mean SFA efficiency per epoch and buoy:\n%s", summary.to_string())

    return df

# ===========================================================================
# Section 4 -- Generation Deficit
# ===========================================================================
def compute_generation_deficit(df: pd.DataFrame, params: Dict) -> pd.DataFrame:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]

    expected_y: np.ndarray = np.exp(beta0 + beta1 * np.log(df[WPF_COL].values + LOG_EPS))
    expected_y = np.clip(expected_y, 0.0, OUTPUT_CAP)

    df = df.copy()
    df["Expected_Y_kW"] = expected_y
    df["Generation_Deficit_kW"] = df["Expected_Y_kW"] - df[TARGET_COL]

    total_deficit_mwh: float = df["Generation_Deficit_kW"].clip(lower=0.0).sum() / 2000.0
    logger.info(
        "Generation deficit computed | mean = %.2f kW | total (positive) = %.1f MWh",
        df["Generation_Deficit_kW"].mean(), total_deficit_mwh
    )
    return df

# ===========================================================================
# Section 5 -- Rolling Aggregation
# ===========================================================================
def aggregate_rolling(df: pd.DataFrame) -> pd.DataFrame:
    df_pivot: pd.DataFrame = df.pivot_table(index=TIMESTAMP_COL, columns=BUOY_COL, values="SFA_Efficiency")
    df_pivot.columns.name = None
    rolling: pd.DataFrame = df_pivot.rolling(ROLLING_WINDOW, min_periods=1).mean()
    logger.info("7-day rolling means computed, shape: %s", rolling.shape)
    return rolling

# ===========================================================================
# Section 6 to 8 -- Visualisations
# ===========================================================================
def _epoch_boundaries(df: pd.DataFrame) -> Dict[int, pd.Timestamp]:
    return {int(epoch): df[df[EPOCH_COL] == epoch][TIMESTAMP_COL].min() for epoch in sorted(df[EPOCH_COL].unique())}

def plot_timeseries(rolling: pd.DataFrame, epoch_bounds: Dict[int, pd.Timestamp], save_path: str) -> None:
    fig, ax = plt.subplots(figsize=(16, 7))
    palette_healthy: List = sns.color_palette("Blues_r", n_colors=len(HEALTHY_BUOYS))
    palette_degraded: List = sns.color_palette("Reds_r", n_colors=len(DEGRADED_BUOYS))

    for i, buoy in enumerate(HEALTHY_BUOYS):
        if buoy in rolling.columns:
            ax.plot(rolling.index, rolling[buoy], color=palette_healthy[i], linewidth=1.4, alpha=0.85, label=buoy.replace("Boia_", "Buoy "))

    for i, buoy in enumerate(DEGRADED_BUOYS):
        if buoy in rolling.columns:
            ax.plot(rolling.index, rolling[buoy], color=palette_degraded[i], linewidth=2.2, alpha=0.95, label=f"{buoy.replace('Boia_', 'Buoy ')} (degraded)", linestyle="--")

    epoch_colors: Dict[int, str] = {1: "#555555", 2: "#E67E22", 3: "#C0392B"}
    epoch_labels: Dict[int, str] = {1: "Epoch 1\n(Golden Period)", 2: "Epoch 2\n(-15% global)", 3: "Epoch 3\n(PTO fault Buoys 9-12)"}
    
    for epoch, ts in epoch_bounds.items():
        ax.axvline(ts, color=epoch_colors[epoch], linestyle=":", linewidth=1.4, alpha=0.7)
        ax.text(ts, 0.04, epoch_labels[epoch], fontsize=14, color=epoch_colors[epoch], ha="left", va="bottom")

    ax.axhspan(0.0, 0.55, alpha=0.07, color="#C0392B", label="Severe degradation zone (<0.55)")
    ax.set_ylim(0.0, 1.08)
    ax.set_ylabel("SFA Technical Efficiency (TE)", fontsize=16)
    ax.set_xlabel("Date", fontsize=16)
    ax.tick_params(axis="y", labelsize=14)
    
    ax.set_title("WEC Phase 2 - SFA Technical Efficiency: 7-Day Rolling Mean\nEpoch 2: Spectral Spreading Penalty | Epoch 3: isolated PTO fault (Buoys 9-12)", fontsize=18, fontweight="bold")
    
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=25, ha="right", fontsize=14)
    ax.legend(fontsize=14, ncol=3, loc="upper right", framealpha=0.9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Timeseries plot saved to: %s", save_path)

def plot_residual_decomposition(df: pd.DataFrame, save_path: str) -> None:
    df_e3: pd.DataFrame = df[df[EPOCH_COL] == 3].copy()
    df_e3["Group"] = df_e3[BUOY_COL].apply(lambda b: "Healthy (Buoys 1-8)" if b in HEALTHY_BUOYS else "Degraded (Buoys 9-12)")
    df_e3["u_hat"] = df_e3["mu_star"].clip(lower=0)
    df_e3["v_hat"] = df_e3["epsilon"] + df_e3["u_hat"]

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    palette_grp: Dict[str, str] = {"Healthy (Buoys 1-8)": COLOR_HEALTHY, "Degraded (Buoys 9-12)": COLOR_DEGRADED}

    ax_left = axes[0]
    for grp, sub in df_e3.groupby("Group"):
        sns.kdeplot(sub["epsilon"], ax=ax_left, label=grp, color=palette_grp[grp], linewidth=2.2, fill=True, alpha=0.25)
    ax_left.axvline(0, color="black", linestyle="--", linewidth=1.2, label="Zero residual")
    ax_left.set_xlabel("Composite Residual epsilon = ln(Y) - frontier", fontsize=16)
    ax_left.set_ylabel("Density", fontsize=16)
    ax_left.set_title("Composite Residual Distribution\nEpoch 3 (by operational group)", fontweight="bold", fontsize=18)
    ax_left.tick_params(axis="both", labelsize=14)
    ax_left.legend(fontsize=14)
    ax_left.grid(True, alpha=0.25)

    ax_right = axes[1]
    for buoy, color, label, col_name in [("Boia_1", COLOR_HEALTHY, "Buoy 1 - noise v (symmetric)", "v_hat"), ("Boia_9", COLOR_DEGRADED, "Buoy 9 - inferred inefficiency u", "u_hat")]:
        sub = df_e3[df_e3[BUOY_COL] == buoy]
        sns.kdeplot(sub[col_name], ax=ax_right, label=label, color=color, linewidth=2.2, fill=True, alpha=0.22)
    ax_right.axvline(0, color="black", linestyle="--", linewidth=1.2)
    ax_right.set_xlabel("Error component magnitude", fontsize=16)
    ax_right.set_ylabel("Density", fontsize=16)
    ax_right.set_title("SFA Error Decomposition: v vs u\nEpoch 3 (representative buoys)", fontweight="bold", fontsize=18)
    ax_right.tick_params(axis="both", labelsize=14)
    ax_right.legend(fontsize=14)
    ax_right.grid(True, alpha=0.25)

    fig.suptitle("SFA Residual Analysis - Epoch 3: Statistical Separation of Noise and Inefficiency", fontsize=20, fontweight="bold")

    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Residual decomposition plot saved to: %s", save_path)

def _compute_frontier_curve(df_epoch: pd.DataFrame, params: Dict, n_points: int = 300) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    beta0: float = params["beta0"]
    beta1: float = params["beta1"]
    sigma_v: float = np.sqrt(params["sigma_v2"])

    wpf_range: np.ndarray = np.linspace(df_epoch[WPF_COL].quantile(0.01), df_epoch[WPF_COL].quantile(0.99), n_points)
    frontier_y: np.ndarray = np.clip(np.exp(beta0 + beta1 * np.log(wpf_range + LOG_EPS)), 0.0, OUTPUT_CAP)
    upper: np.ndarray = np.clip(frontier_y * np.exp(+sigma_v), 0.0, OUTPUT_CAP)
    lower: np.ndarray = np.clip(frontier_y * np.exp(-sigma_v), 0.0, OUTPUT_CAP)
    return wpf_range, frontier_y, lower, upper

def _draw_frontier_overlay(ax: plt.Axes, wpf_range: np.ndarray, frontier_y: np.ndarray, lower: np.ndarray, upper: np.ndarray, show_legend_label: bool = True) -> None:
    lbl_frontier = r"SFA Deterministic Frontier: $\hat{Y}=\exp(\hat\beta_0+\hat\beta_1\ln WPF)$" if show_legend_label else "_nolegend_"
    lbl_band = r"$\pm1\,\sigma_v$ stochastic band (noise scatter)" if show_legend_label else "_nolegend_"
    ax.plot(wpf_range, frontier_y, color=COLOR_FRONTIER, linewidth=2.2, linestyle="-", label=lbl_frontier, zorder=5)
    ax.fill_between(wpf_range, lower, upper, alpha=0.12, color=COLOR_HEALTHY, label=lbl_band)


def _buoy_color(buoy: str) -> str:
    return COLOR_DEGRADED if buoy in DEGRADED_BUOYS else COLOR_HEALTHY


def plot_triple_frontier(df: pd.DataFrame, params: Dict, save_path: str, epoch: int = 3, scatter_sample_per_buoy: int = 120) -> None:
    logger.info("Building triple frontier scatter for Epoch %d", epoch)
    df_epoch: pd.DataFrame = df[df[EPOCH_COL] == epoch].copy()
    wpf_range, frontier_y, lower, upper = _compute_frontier_curve(df_epoch, params)

    epoch_start: pd.Timestamp = df_epoch[TIMESTAMP_COL].min()
    epoch_end: pd.Timestamp = df_epoch[TIMESTAMP_COL].max()
    epoch_mid: pd.Timestamp = epoch_start + (epoch_end - epoch_start) / 2

    ts_per_buoy: List[set] = [set(df_epoch[df_epoch[BUOY_COL] == b][TIMESTAMP_COL].tolist()) for b in ALL_BUOYS if b in df_epoch[BUOY_COL].unique()]
    
    if ts_per_buoy:
        common_ts: set = ts_per_buoy[0].intersection(*ts_per_buoy[1:])
    else:
        common_ts = set()

    snapshot_ts: Optional[pd.Timestamp] = None
    if common_ts:
        common_series: pd.Series = pd.Series(sorted(common_ts))
        snapshot_ts = common_series.iloc[(common_series - epoch_mid).abs().argmin()]
        logger.info("Snapshot timestamp (Panel B): %s", snapshot_ts)
    else:
        logger.warning("No common timestamp found for all buoys in Epoch %d - Panel B will be empty", epoch)
    
    df_mean: pd.DataFrame = df_epoch.groupby(BUOY_COL)[[WPF_COL, TARGET_COL, "Expected_Y_kW", "Generation_Deficit_kW", "SFA_Efficiency"]].mean().reindex(ALL_BUOYS).dropna()

    fig, axes = plt.subplots(1, 3, figsize=(20, 9.5), sharex=True, sharey=True)
    panel_titles: List[str] = [
        f"A - Population View\n(sampled 30-min observations, Epoch {epoch})",
        f"B - Instantaneous Snapshot\n(single common timestamp: {snapshot_ts.strftime('%Y-%m-%d %H:%M') if snapshot_ts else 'N/A'})",
        f"C - Mean Operating Point\n(per-buoy epoch mean)"
    ]

    # Panel A
    ax_a: plt.Axes = axes[0]
    sampled_chunks = []
    for _, group_data in df_epoch.groupby(BUOY_COL):
        n_samples = min(len(group_data), scatter_sample_per_buoy)
        sampled_chunks.append(group_data.sample(n=n_samples, random_state=42))
    df_sample_a: pd.DataFrame = pd.concat(sampled_chunks, ignore_index=True)

    for buoy in ALL_BUOYS:
        sub = df_sample_a[df_sample_a[BUOY_COL] == buoy]
        if not sub.empty:
            buoy_label = buoy.replace("Boia_", "Buoy ")
            ax_a.scatter(sub[WPF_COL], sub[TARGET_COL], color=_buoy_color(buoy), alpha=0.25, s=14, edgecolors="none", label=buoy_label)
    _draw_frontier_overlay(ax_a, wpf_range, frontier_y, lower, upper, show_legend_label=True)
    ax_a.set_title(panel_titles[0], fontsize=18, fontweight="bold")
    ax_a.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_a.set_ylabel("Energy Generation [kW]", fontsize=18)
    ax_a.tick_params(labelsize=16)
    ax_a.grid(True, alpha=0.2)

    # Panel B
    ax_b: plt.Axes = axes[1]
    _draw_frontier_overlay(ax_b, wpf_range, frontier_y, lower, upper, show_legend_label=False)
    if snapshot_ts is not None:
        df_snap: pd.DataFrame = df_epoch[df_epoch[TIMESTAMP_COL] == snapshot_ts]
        for buoy in ALL_BUOYS:
            row = df_snap[df_snap[BUOY_COL] == buoy]
            if not row.empty:
                x_val, y_val = row[WPF_COL].values[0], row[TARGET_COL].values[0]
                y_exp, deficit = row["Expected_Y_kW"].values[0], row["Generation_Deficit_kW"].values[0]
                buoy_label = buoy.replace("Boia_", "Buoy ")
                
                if buoy in DEGRADED_BUOYS:
                    ax_b.vlines(x=x_val, ymin=y_val, ymax=y_exp, color=COLOR_DEGRADED, linestyle='--', linewidth=1.5, zorder=5)
                    ax_b.text(x_val + 0.5, (y_val + y_exp) / 2, f"-{deficit:.0f} kW", color=COLOR_DEGRADED, fontsize=15, fontweight="bold", va='center')
                ax_b.scatter(x_val, y_val, color=_buoy_color(buoy), s=120, alpha=0.92, edgecolors="white", linewidths=0.8, zorder=6, label=buoy_label)
                buoy_idx: str = buoy.split("_")[-1]
                ax_b.annotate(buoy_idx, xy=(x_val, y_val), xytext=(3, 4), textcoords="offset points", fontsize=14, color=_buoy_color(buoy), fontweight="bold")
    ax_b.set_title(panel_titles[1], fontsize=18, fontweight="bold")
    ax_b.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_b.tick_params(labelsize=16)
    ax_b.grid(True, alpha=0.2)

    # Panel C
    ax_c: plt.Axes = axes[2]
    _draw_frontier_overlay(ax_c, wpf_range, frontier_y, lower, upper, show_legend_label=False)
    best_buoy, worst_buoy = df_mean["SFA_Efficiency"].idxmax(), df_mean["SFA_Efficiency"].idxmin()
    for buoy in df_mean.index:
        row_wpf, row_y = df_mean.loc[buoy, WPF_COL], df_mean.loc[buoy, TARGET_COL]
        buoy_label = buoy.replace("Boia_", "Buoy ")
        ax_c.scatter(row_wpf, row_y, marker="X", color=_buoy_color(buoy), s=160, alpha=0.95, edgecolors="white", linewidths=0.8, zorder=6, label=buoy_label)
        
        if buoy == worst_buoy:
            deficit, te = df_mean.loc[buoy, "Generation_Deficit_kW"], df_mean.loc[buoy, "SFA_Efficiency"]
            bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec=COLOR_DEGRADED, lw=1.2, alpha=0.9)
            # Corrigido o \\n para \n e alterado a label para usar Buoy
            ax_c.annotate(f"Worst Asset ({buoy_label})\nTE: {te:.2f} | Deficit: -{deficit:.1f} kW", xy=(row_wpf, row_y), xytext=(row_wpf + 2, row_y - 50), arrowprops=dict(facecolor=COLOR_DEGRADED, shrink=0.05, width=1.5, headwidth=5), fontsize=13, fontweight="bold", color=COLOR_DEGRADED, bbox=bbox_props, zorder=10)
        elif buoy == best_buoy:
            te = df_mean.loc[buoy, "SFA_Efficiency"]
            bbox_props = dict(boxstyle="round,pad=0.3", fc="white", ec=COLOR_HEALTHY, lw=1.2, alpha=0.9)
            # Corrigido o \\n para \n e alterado a label para usar Buoy
            ax_c.annotate(f"Best Asset ({buoy_label})\nTE: {te:.2f}", xy=(row_wpf, row_y), xytext=(row_wpf - 18, row_y + 40), arrowprops=dict(facecolor=COLOR_HEALTHY, shrink=0.05, width=1.5, headwidth=5), fontsize=13, fontweight="bold", color=COLOR_HEALTHY, bbox=bbox_props, zorder=10)
        else:
            buoy_idx = buoy.split("_")[-1]
            ax_c.annotate(buoy_idx, xy=(row_wpf, row_y), xytext=(4, 4), textcoords="offset points", fontsize=14, color=_buoy_color(buoy), fontweight="bold")
    ax_c.set_title(panel_titles[2], fontsize=18, fontweight="bold")
    ax_c.set_xlabel("Wave Power Flux (WPF) [kW/m]", fontsize=18)
    ax_c.tick_params(labelsize=16)
    ax_c.grid(True, alpha=0.2)

    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    legend_elements = [
        Patch(facecolor=COLOR_HEALTHY,  label="Healthy fleet (Buoys 1-8)"),
        Patch(facecolor=COLOR_DEGRADED, label="Degraded fleet (Buoys 9-12)"),
        Line2D([0], [0], color=COLOR_FRONTIER, linewidth=2, label="SFA Deterministic Frontier"),
        Patch(facecolor=COLOR_HEALTHY, alpha=0.20, label=r"$\pm1\,\sigma_v$ Stochastic Band"),
    ]
    
    
    # Subtitulo redundante removido e atualizado
    fig.suptitle(f"WEC Phase 2 - SFA Triple Frontier Analysis (Epoch {epoch})", fontsize=18, fontweight="bold", y=1.01)

    fig.legend(handles=legend_elements, loc="upper center", ncol=4, fontsize=18, framealpha=0.9, bbox_to_anchor=(0.5, 0.01))
    
    plt.tight_layout()
    plt.savefig(save_path, dpi=600, bbox_inches="tight")
    plt.close(fig)
    logger.info("Triple frontier scatter saved to: %s", save_path)

# ===========================================================================
# Section 9 & 10 -- Reporting and Export
# ===========================================================================
def print_degradation_report(df: pd.DataFrame) -> None:
    separator: str = "=" * 72
    pivot: pd.DataFrame = (
        df.groupby([EPOCH_COL, BUOY_COL])["SFA_Efficiency"]
        .mean()
        .unstack(EPOCH_COL)
        .rename(columns={
            1: "Epoch1_base",
            2: "Epoch 2_Sub_optimal_Spectrum",
            3: "Epoch3_fault",
        })
    )
    pivot["Delta_E1_to_E3"] = pivot["Epoch3_fault"] - pivot["Epoch1_base"]
    pivot["Status"] = pivot["Delta_E1_to_E3"].apply(
        lambda d: "DEGRADATION DETECTED" if d < -0.20 else "normal"
    )

    logger.info("\n" + separator)
    logger.info("SFA DEGRADATION REPORT -- Phase 2 Summary")
    logger.info(separator)
    logger.info("Part 1 -- Cross-Epoch Efficiency Summary:")
    logger.info("\n" + pivot.round(4).to_string())
    logger.info(separator)

    df_e3: pd.DataFrame = df[df[EPOCH_COL] == 3].copy()
    deficit_summary: pd.DataFrame = (
        df_e3.groupby(BUOY_COL)
        .agg(
            Mean_SFA_Efficiency=("SFA_Efficiency", "mean"),
            Mean_Deficit_kW=("Generation_Deficit_kW", "mean"),
        )
        .reindex(ALL_BUOYS)
        .dropna()
        .sort_values("Mean_Deficit_kW", ascending=False)
        .round(4)
    )

    logger.info("Part 2 -- Epoch 3 Asset Criticality Ranking (descending deficit):")
    logger.info("\n" + deficit_summary.to_string())
    logger.info(separator)

    for buoy, row in deficit_summary.iterrows():
        logger.info(
            "%s | Mean SFA Efficiency = %.4f | Mean Generation Deficit = %.2f kW",
            str(buoy).replace("_", " "), row["Mean_SFA_Efficiency"], row["Mean_Deficit_kW"]
        )

    worst_buoy: str = str(deficit_summary.index[0])
    worst_deficit: float = float(deficit_summary.iloc[0]["Mean_Deficit_kW"])
    worst_efficiency: float = float(deficit_summary.iloc[0]["Mean_SFA_Efficiency"])

    logger.info(separator)
    logger.info("CONCLUSION -- Worst performing asset: %s", worst_buoy.replace("_", " "))
    logger.info("  Mean SFA Technical Efficiency : %.4f (%.1f%% of frontier)", worst_efficiency, worst_efficiency * 100.0)
    logger.info("  Mean Generation Deficit        : %.2f kW per observation", worst_deficit)
    logger.info("Recommendation: prioritise inspection and PTO maintenance of %s.", worst_buoy.replace("_", " "))
    logger.info(separator + "\n")

def export_artefacts(df: pd.DataFrame) -> None:
    logger.info("Exporting intermediate artefacts for Phase 3")
    export_cols = [TIMESTAMP_COL, BUOY_COL, EPOCH_COL, "SFA_Efficiency", "Generation_Deficit_kW"]
    df_export = df[export_cols].copy()
    
    os.makedirs(os.path.dirname(PHASE2_CSV_OUT), exist_ok=True)
    df_export.to_csv(PHASE2_CSV_OUT, index=False)
    logger.info("Phase 2 data contract exported to: %s", PHASE2_CSV_OUT)

## Execution Block
Chamada sequencial das funções para correr a Fase 2. Inclui a geração de dados sintéticos caso o ficheiro CSV original não exista.

In [3]:
logger.info("=" * 72)
logger.info("WEC Phase 2 -- Stochastic Frontier Analysis (SFA)")
logger.info("=" * 72)

PLOT_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------------
# Stage 1 -- Load or Generate Mock Data
# ------------------------------------------------------------------
if os.path.exists(DATA_PATH):
    df: pd.DataFrame = load_and_prepare(DATA_PATH)
else:
    logger.warning("CSV not found at '%s' -- generating synthetic dataset", DATA_PATH)
    rng = np.random.default_rng(42)
    n_per_buoy: int = 2400
    records: List[pd.DataFrame] = []
    
    for buoy in ALL_BUOYS:
        for epoch in [1, 2, 3]:
            hs: np.ndarray = rng.uniform(0.8, 5.0, n_per_buoy)
            te: np.ndarray = rng.uniform(5.0, 15.0, n_per_buoy)
            wpf: np.ndarray = 0.49 * hs ** 2 * te
            base_eff: float = 1.0 if epoch == 1 else 0.85 if epoch == 2 else (0.45 if buoy in DEGRADED_BUOYS else 0.90)
            energy: np.ndarray = (base_eff * 2.8 * wpf + rng.normal(0, 12, n_per_buoy)).clip(1.0, OUTPUT_CAP)
            start: pd.Timestamp = pd.Timestamp(f"2025-0{2 + epoch}-01")
            ts: pd.DatetimeIndex = pd.date_range(start, periods=n_per_buoy, freq="30min")

            records.append(pd.DataFrame({
                TIMESTAMP_COL: ts,
                BUOY_COL:      buoy,
                "Hs__m":       hs,
                "Te__s":       te,
                WPF_COL:       wpf,
                TARGET_COL:    energy,
                EPOCH_COL:     epoch,
            }))

    df_raw: pd.DataFrame = pd.concat(records, ignore_index=True)
    os.makedirs("dataset2", exist_ok=True)
    df_raw.to_csv(DATA_PATH, index=False)
    logger.info("Synthetic CSV saved to: %s", DATA_PATH)
    df = load_and_prepare(DATA_PATH)

# ------------------------------------------------------------------
# Execution Pipeline
# ------------------------------------------------------------------
params: Dict = fit_sfa_epoch1(df)
df = score_efficiency(df, params)
df = compute_generation_deficit(df, params)

rolling: pd.DataFrame = aggregate_rolling(df)
print_degradation_report(df)

epoch_bounds: Dict[int, pd.Timestamp] = _epoch_boundaries(df)

plot_timeseries(rolling, epoch_bounds, save_path=str(PLOT_DIR / "wec_phase2_SFA_sfa_timeseries.png"))
plot_residual_decomposition(df, save_path=str(PLOT_DIR / "wec_phase2_SFA_sfa_residuals.png"))
plot_triple_frontier(df, params, save_path=str(PLOT_DIR / "wec_phase2_sfa_triple_frontier.png"), epoch=3)

export_artefacts(df)

logger.info("Phase 2 SFA complete. Output files written to: %s", PLOT_DIR.resolve())
logger.info("=" * 72)

# df.head() # Descomentar para visualizar os dados

2026-06-09 16:52:04 | INFO | ========================================================================
2026-06-09 16:52:04 | INFO | WEC Phase 2 -- Stochastic Frontier Analysis (SFA)
2026-06-09 16:52:04 | INFO | ========================================================================
2026-06-09 16:52:04 | INFO | Loading data from: dataset2/wec_c5_mock_data_epochs.csv
2026-06-09 16:52:04 | INFO | Raw shape: (86976, 17)
2026-06-09 16:52:04 | WARNING | Dropped 854 rows with non-positive WPF or output
2026-06-09 16:52:04 | INFO | Prepared shape: (86122, 19) | Epochs present: [np.int64(1), np.int64(2), np.int64(3)]
2026-06-09 16:52:04 | INFO | Fitting SFA on Epoch 1 Ramp-up Region: 68955 observations from 12 buoys
2026-06-09 16:52:06 | INFO | MLE results (Epoch 1 frontier):
2026-06-09 16:52:06 | INFO |   beta_0    = +2.085329
2026-06-09 16:52:06 | INFO |   beta_1    = +0.828365  (output elasticity)
2026-06-09 16:52:06 | INFO |   lambda    = 4.352400   (sigma_u / sigma_v)
2026-06-09 16:52:06 |